# Introdução

O objetivo desta análise é explorar um banco de dados de uma plataforma digital de leitura que reúne informações sobre livros, autores, editoras, avaliações e resenhas de usuários. A proposta é compreender melhor o catálogo disponível e o comportamento dos leitores por meio de consultas SQL.

Para isso, foi realizada uma exploração inicial das tabelas e da qualidade dos dados, seguida pela análise de métricas relevantes, como quantidade de livros, avaliações, desempenho de autores e nível de engajamento dos usuários. Esses resultados servirão como base para gerar insights que possam apoiar a criação de uma proposta de valor para um novo produto.

### Importando bibliotecas

In [1]:

import pandas as pd
from sqlalchemy import create_engine
db_config = {
 'user': 'practicum_student', # username
 'pwd': 'QnmDH8Sc2TQLvy2G3Vvh7', # password
 'host': 'yp-trainers-practicum.cluster-czs0gxyx2d8w.us-east-1.rds.amazonaws.com',
 'port': 5432, # connection port
 'db': 'data-analyst-final-project-db' # the name of the database
 }
connection_string = 'postgresql://{}:{}@{}:{}/{}'.format(db_config['user'],
db_config['pwd'],
db_config['host'],
db_config['port'],

db_config['db'])

#Conectando-se ao Banco de Dados 2
engine = create_engine(connection_string, connect_args={'sslmode':'require'})

query = "SELECT 1;"
pd.read_sql(query, con=engine)



,?column?
0,1


### Explorar a estrutura do banco de dados

In [2]:

query = """
SELECT table_name
FROM information_schema.tables
WHERE table_schema = 'public';
"""

pd.read_sql(query, engine)




,table_name
0,ratings
1,advertisment_costs
2,authors
3,orders
4,reviews
5,visits
6,books
7,users
8,publishers


### Explorar as tabelas

In [3]:
# explorar as tabelas
def q(sql):
    return pd.read_sql(sql, engine)

#books
q("""
SELECT column_name, data_type
FROM information_schema.columns
WHERE table_name = 'books'
ORDER BY ordinal_position;
""")

#authors
q("""
SELECT column_name, data_type
FROM information_schema.columns
WHERE table_name = 'authors'
ORDER BY ordinal_position;
""")


#publishers
q("""
SELECT column_name, data_type
FROM information_schema.columns
WHERE table_name = 'publishers'
ORDER BY ordinal_position;
""")

#ratings
q("""
SELECT column_name, data_type
FROM information_schema.columns
WHERE table_name = 'ratings'
ORDER BY ordinal_position;
""")

#reviews
q("""
SELECT column_name, data_type
FROM information_schema.columns
WHERE table_name = 'reviews'
ORDER BY ordinal_position;
""")


q("SELECT * FROM books LIMIT 5;")
q("SELECT * FROM authors LIMIT 5;")
q("SELECT * FROM publishers LIMIT 5;")
q("SELECT * FROM ratings LIMIT 5;")
q("SELECT * FROM reviews LIMIT 5;")


,review_id,book_id,username,text
0,1,1,brandtandrea,Mention society tell send professor analysis. ...
1,2,1,ryanfranco,Foot glass pretty audience hit themselves. Amo...
2,3,2,lorichen,Listen treat keep worry. Miss husband tax but ...
3,4,3,johnsonamanda,Finally month interesting blue could nature cu...
4,5,3,scotttamara,Nation purpose heavy give wait song will. List...


### Data Quality Check — Books

In [4]:
#tamamho da tabela
q("SELECT COUNT(*) FROM books;")


,count
0,1000


In [5]:
#checar valores nulos
q("""
SELECT COUNT(*)
FROM books
WHERE publication_date IS NULL;
""")


,count
0,0


In [6]:
#checar datas estranhas
q("""
SELECT MIN(publication_date),
       MAX(publication_date)
FROM books;
""")


,min,max
0,1952-12-01,2020-03-31


In [7]:
#ver se existe livros sem data
q("""
SELECT COUNT(*) AS null_publication_date
FROM books
WHERE publication_date IS NULL;
""")


,null_publication_date
0,0


In [8]:
#checar se tem book_id duplicados
q("""
SELECT COUNT(*) AS duplicated_book_ids
FROM (
    SELECT book_id
    FROM books
    GROUP BY book_id
    HAVING COUNT(*) > 1
) t;
""")


,duplicated_book_ids
0,0


A tabela books foi analisada para verificar possíveis inconsistências.
Não foram encontrados IDs duplicados ou valores nulos, indicando boa integridade dos dados.

### Data Quality Check — Authors

In [9]:

#tamanho da tabela
q("SELECT COUNT(*) FROM authors;")


,count
0,636


In [10]:
#ver duplicados
q("""
SELECT author_id, COUNT(*)
FROM authors
GROUP BY author_id
HAVING COUNT(*) > 1;
""")


,author_id,count


In [11]:

#checar valores nulos
q("""
SELECT COUNT(*) AS null_author_names
FROM authors
WHERE author IS NULL;
""")



,null_author_names
0,0


A tabela authors foi analisada para verificar possíveis inconsistências.
Não foram encontrados IDs duplicados ou valores nulos, indicando boa integridade dos dados.

### Data Quality Check — Publishers

In [12]:
#tamanho da tabela
q("SELECT COUNT(*) FROM publishers;")

,count
0,340


In [13]:
#ver duplicados

q("""
SELECT publisher_id, COUNT(*) AS cnt
FROM publishers
GROUP BY publisher_id
HAVING COUNT(*) > 1;
""")


,publisher_id,cnt


In [14]:
#checar valores nulos
q("""
SELECT COUNT(*) AS null_author_names
FROM authors
WHERE author IS NULL;
""")



,null_author_names
0,0


### Data Quality Check — Ratings

In [15]:
#tamaho da tabela
q("SELECT COUNT(*) FROM ratings;")


,count
0,6456


In [16]:
#range de notas
q("""
SELECT MIN(rating), MAX(rating)
FROM ratings;
""")


,min,max
0,1,5


In [17]:
#usuários avaliando o mesmo livro

q("""
SELECT username, book_id, COUNT(*)
FROM ratings
GROUP BY username, book_id
HAVING COUNT(*) > 1;
""")


,username,book_id,count


### Data Quality Check — Reviews

In [18]:

q("""
SELECT
COUNT(*) AS total_reviews,
SUM(CASE WHEN text IS NULL THEN 1 ELSE 0 END) AS null_reviews,
SUM(CASE WHEN TRIM(text) = '' THEN 1 ELSE 0 END) AS empty_reviews
FROM reviews;
""")


,total_reviews,null_reviews,empty_reviews
0,2793,0,0


<div class="alert alert-block alert-success">
<b> Comentário: </b> <a class="tocSkip"></a>

    
- As tabelas foram lidas corretamente.
- Gostei da ideia de quality check!
</div>

O banco de dados é composto por tabelas que armazenam informações sobre livros, autores, editoras e interações dos usuários.

A tabela books atua como a tabela central do modelo, contendo os dados principais de cada livro. Ela se conecta à tabela authors por meio da coluna author_id, permitindo identificar o autor de cada obra, e à tabela publishers através de publisher_id, que indica a editora responsável pela publicação.

As tabelas ratings e reviews registram as interações dos usuários com os livros. A tabela ratings armazena as notas atribuídas, enquanto reviews contém as resenhas escritas. Ambas se relacionam com books pela coluna book_id, possibilitando analisar a recepção e o engajamento dos leitores.

De modo geral, o modelo permite combinar informações cadastrais dos livros com o comportamento dos usuários, viabilizando análises sobre popularidade, avaliação e participação dos leitores.

### Quantos livros foram prublicados depois de 1º de janeiro de 2000?

In [19]:
#quantos livros foram prublicados depois de 1º de janeiro de 2000?
q("""
SELECT COUNT(*)
FROM books
WHERE publication_date > '2000-01-01';
""")

,count
0,819


<div class="alert alert-block alert-success">
<b> Comentário: </b> <a class="tocSkip"></a>

    
- A quantidade de livros lançadas após Jan-2000 foi calculada

</div>

### Encontre o número de avaliações e a classificação média para cada livro.


In [20]:
#quantas linhas de rating existem por livro

q("""
SELECT
  book_id,
  COUNT(*) AS rating_count,
  AVG(rating) AS avg_rating
FROM ratings
GROUP BY book_id
ORDER BY rating_count DESC
LIMIT 10;
""").reset_index(drop=True)


,book_id,rating_count,avg_rating
0,948,160,3.662500
1,750,88,4.125000
2,673,86,3.825581
3,75,84,3.678571
4,302,82,4.414634
5,299,80,4.287500
6,301,75,4.186667
7,722,74,4.391892
8,79,74,3.729730
9,300,73,4.246575


In [21]:
#quantas avaliações por livro

q("""
SELECT
  b.book_id,
  b.title,
  COUNT(r.rating_id) AS rating_count,
  AVG(r.rating) AS avg_rating
FROM books b
LEFT JOIN ratings r ON r.book_id = b.book_id
GROUP BY b.book_id, b.title
ORDER BY rating_count DESC, avg_rating DESC
LIMIT 20;
""").reset_index(drop=True)


,book_id,title,rating_count,avg_rating
0,948,Twilight (Twilight #1),160,3.662500
1,750,The Hobbit or There and Back Again,88,4.125000
2,673,The Catcher in the Rye,86,3.825581
3,75,Angels & Demons (Robert Langdon #1),84,3.678571
4,302,Harry Potter and the Prisoner of Azkaban (Harr...,82,4.414634
5,299,Harry Potter and the Chamber of Secrets (Harry...,80,4.287500
6,301,Harry Potter and the Order of the Phoenix (Har...,75,4.186667
7,722,The Fellowship of the Ring (The Lord of the Ri...,74,4.391892
8,79,Animal Farm,74,3.729730
9,300,Harry Potter and the Half-Blood Prince (Harry ...,73,4.246575


### Identifique a editora que lançou o maior número de livros com mais de 50 páginas

In [22]:
#Identifique a editora que lançou o maior número de livros com mais de 50 páginas

q("""
WITH counts AS (
  SELECT
    p.publisher,
    COUNT(*) AS num_books_over_50_pages
  FROM books b
  JOIN publishers p ON p.publisher_id = b.publisher_id
  WHERE b.num_pages > 50
  GROUP BY p.publisher
)
SELECT *
FROM counts
WHERE num_books_over_50_pages = (SELECT MAX(num_books_over_50_pages) FROM counts);
""")


,publisher,num_books_over_50_pages
0,Penguin Books,42


### Identifique o autor com a média mais alta de classificação de livros: olhe apenas para livros com pelo menos 50 classificações.

In [23]:
#Identificar o autor com a média mais alta de classificação de livros: olhe apenas para livros com pelo menos 50 classificações

#quais livros têm  50 ou mais de avaliações

q("""
SELECT COUNT(*) AS num_books_with_50plus_ratings
FROM (
  SELECT book_id
  FROM ratings
  GROUP BY book_id
  HAVING COUNT(*) >= 50
) t;
""")



,num_books_with_50plus_ratings
0,19


In [24]:
q("""
SELECT book_id, COUNT(*) AS n_ratings
FROM ratings
GROUP BY book_id
HAVING COUNT(*) >= 50;
""").reset_index(drop=True)


,book_id,n_ratings
0,75,84
1,750,88
2,545,66
3,948,160
4,488,61
5,696,59
6,722,74
7,627,57
8,733,56
9,779,62


In [25]:
#calcular média livros com avaliação com 50 ou mais

q("""
WITH books_50 AS (
    SELECT book_id
    FROM ratings
    GROUP BY book_id
    HAVING COUNT(*) >= 50
),
author_avg AS (
    SELECT
        a.author_id,
        a.author,
        AVG(r.rating) AS avg_rating,
        COUNT(r.rating_id) AS total_ratings
    FROM books b
    JOIN books_50 bf ON bf.book_id = b.book_id
    JOIN ratings r ON r.book_id = b.book_id
    JOIN authors a ON a.author_id = b.author_id
    GROUP BY a.author_id, a.author
)
SELECT *
FROM author_avg
ORDER BY avg_rating DESC, total_ratings DESC
LIMIT 1;
""").reset_index(drop=True)


,author_id,author,avg_rating,total_ratings
0,236,J.K. Rowling/Mary GrandPré,4.287097,310


<div class="alert alert-block alert-success">
<b> Comentário: </b> <a class="tocSkip"></a>
    
Análise das editoras e das classificações foram realizadas corretamente

</div>

Ao aplicar o critério de incluir apenas livros com pelo menos 50 avaliações, foram identificados 19 livros elegíveis. Considerando apenas esses livros, o autor com maior nota média foi J.K. Rowling/Mary GrandPré, com média de ≈ 4.29 (baseada em 310 avaliações). Esse critério reduz a influência de livros com poucas avaliações, tornando a comparação mais confiável.

### Encontre o número médio de resenhas com texto entre usuários que avaliaram mais de 50 livros.

In [27]:
q("""
WITH heavy_raters AS (
    SELECT username
    FROM ratings
    GROUP BY username
    HAVING COUNT(DISTINCT book_id) > 50
),
reviews_per_user AS (
    SELECT
        h.username,
        COUNT(r.review_id) AS text_reviews
    FROM heavy_raters h
    LEFT JOIN reviews r ON r.username = h.username
       AND r.text IS NOT NULL
       AND TRIM(r.text) <> ''
    GROUP BY h.username
)
SELECT AVG(text_reviews) AS avg_text_reviews_among_heavy_raters
FROM reviews_per_user;
""")


,avg_text_reviews_among_heavy_raters
0,24.333333


Usuários que avaliaram mais de 50 livros escreveram, em média, aproximadamente 24 resenhas com texto. Isso indica que leitores mais ativos tendem também a contribuir com conteúdo escrito, embora o número de resenhas seja inferior ao número de avaliações.

<div class="alert alert-block alert-success">
<b> Comentário: </b> <a class="tocSkip"></a>
    
Todas tarefas realizadas corretamente :) Show!

</div>